# GBT macro-source comparison

Compare the project baseline FRED-MD macro panel against the balanced real-time panel using the same target, split, and GBT feature recipe.

In [ ]:
from itertools import product

import pandas as pd

TARGET_MODE = "annual_overlapping"  # "annual_overlapping", "annual_nonoverlapping", or "monthly"
TARGET_MATURITY = "24"
OOS_START = pd.Timestamp("1989-01-31")  # change as needed
ENGINE = "lgbm"  # "xgb" or "lgbm"

FEATURE_SPECS = {
    "project_pca_blocks": {
        "fred": {"method": "pca", "n_components": 8},
        "forward": {"method": "raw"},
        "yields": {"method": "pca", "n_components": 3},
    },
    "bianchi_raw_blocks": {
        "fred": {"method": "raw"},
        "forward": {"method": "raw"},
        "yields": {"method": "raw"},
    },
}
FEATURE_METHODS = ("project_pca_blocks", "bianchi_raw_blocks")

MACRO_SOURCES = ("current_fred_md", "realtime_balanced")

LGBM_DOC_DEFAULTS = {
    "num_leaves": 31,
    "max_depth": -1,
    "learning_rate": 0.1,
    "n_estimators": 100,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_data_in_leaf": 20,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
}

LGBM_NUM_THREADS = 6
LGBM_FORCE_ROW_WISE = True

LGBM_REG_GRID = [
    {
        "colsample_bytree": colsample_bytree,
        "min_data_in_leaf": min_data_in_leaf,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
    }
    for colsample_bytree, min_data_in_leaf, reg_alpha, reg_lambda in product(
        (0.3, 0.4, 0.5),
        (15, 20, 30),
        (0.0, 0.5),
        (1.0, 2.0, 5.0),
    )
]


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").exists() and (ROOT.parent / "utils").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import utils.base_utils as bu
import utils.window_utils as wu
from models.gbt import LightGBMModel, XGBoostModel
from utils.macro_grouping import add_group_level, build_full_group_mapping

start_date = "1971-08-31"
end_date = "2018-12-31"
DROP_SERIES = ("HWI", "HWIURATIO")
IMPUTE_STRATEGY = "median"

for feature_method in FEATURE_METHODS:
    if feature_method not in FEATURE_SPECS:
        raise ValueError(f"Unknown feature method: {feature_method}")

yearly_maturities = [str(i) for i in range(12, 121) if i % 12 == 0]
yearly_yields = bu.get_yields(type="lw", start=start_date, end=end_date, maturities=yearly_maturities)
forward = bu.get_forward_rates(yearly_yields)
xr = bu.get_excess_returns(yearly_yields, horizon=12).dropna()

monthly_maturities = [str(i) for i in range(1, 121)]
monthly_yields = bu.get_yields(type="lw", start=start_date, end=end_date, maturities=monthly_maturities)
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna()

fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
current_macro = bu.get_fred_data("data/2026-01-MD.csv", start=fred_md_start_date, end=end_date).shift(1)
realtime_macro = bu.get_realtime_fred_data(start=fred_md_start_date, end=end_date)

macro_panels = {
    "current_fred_md": bu.prepare_macro_panel_for_project(
        current_macro,
        extra_drop_series=DROP_SERIES,
    ),
    "realtime_balanced": bu.prepare_macro_panel_for_project(
        realtime_macro,
        extra_drop_series=DROP_SERIES,
    ),
}

target_specs = {
    "annual_overlapping": {"frame": xr, "gap": 0},
    "annual_nonoverlapping": {"frame": xr, "gap": 11},
    "monthly": {"frame": monthly_xr, "gap": 0},
}
if TARGET_MODE not in target_specs:
    raise ValueError(f"Unknown TARGET_MODE: {TARGET_MODE}")
if ENGINE not in {"xgb", "lgbm"}:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")
unknown_sources = [name for name in MACRO_SOURCES if name not in macro_panels]
if unknown_sources:
    raise ValueError(f"Unknown macro sources: {unknown_sources}")

target_frame = target_specs[TARGET_MODE]["frame"]
WINDOW_GAP = target_specs[TARGET_MODE]["gap"]
if TARGET_MATURITY not in target_frame.columns:
    raise ValueError(f"Maturity {TARGET_MATURITY} not available for mode {TARGET_MODE}")

common_dates = target_frame.index.intersection(yearly_yields.index).intersection(forward.index)
for panel in macro_panels.values():
    common_dates = common_dates.intersection(panel.index)
common_dates = common_dates.sort_values()

yearly_yields_model = yearly_yields.loc[common_dates]
forward_model = forward.loc[common_dates]
macro_panels = {name: panel.loc[common_dates] for name, panel in macro_panels.items()}
y = target_frame.loc[common_dates, TARGET_MATURITY].values
dates = common_dates

print(f"Mode: {TARGET_MODE}, maturity: {TARGET_MATURITY}m, engine: {ENGINE}")
print(f"Gap: {WINDOW_GAP}")
print(f"OOS start: {OOS_START.date()}")
print(f"Shared sample rows: {len(dates)}")
print(f"Macro sources: {', '.join(MACRO_SOURCES)}")
print(f"Feature methods: {', '.join(FEATURE_METHODS)}")
print(f"LightGBM docs defaults: {LGBM_DOC_DEFAULTS}")
print(f"LightGBM regularization grid size: {len(LGBM_REG_GRID)}")
print(f"LightGBM threads (n_jobs): {LGBM_NUM_THREADS}")
print(f"LightGBM force_row_wise: {LGBM_FORCE_ROW_WISE}")
print("Dropped macro series for fairness: HWI, HWIURATIO")
print("Residual missing values are handled with leakage-safe median imputation inside the GBT wrapper.")


In [ ]:
def build_model(engine, feature_spec):
    common_kwargs = {
        "features": feature_spec,
        "tune_every": 60,
        "impute_strategy": IMPUTE_STRATEGY,
    }
    if engine == "xgb":
        return XGBoostModel(**common_kwargs)
    return LightGBMModel(
        **common_kwargs,
        **LGBM_DOC_DEFAULTS,
        arch_grid=LGBM_REG_GRID,
        n_jobs=LGBM_NUM_THREADS,
        force_row_wise=LGBM_FORCE_ROW_WISE,
        verbose=-1,
    )


def build_feature_frame(fred_md):
    series_to_group = build_full_group_mapping(fred_md, forward_model, yearly_yields_model)
    X_source = pd.concat(
        [fred_md, forward_model, yearly_yields_model],
        axis=1,
        keys=["fred", "forward", "yields"],
    )
    X_source = add_group_level(X_source, series_to_group, level_name="group")
    return X_source.sort_index(axis=1, level="group")


def effective_feature_count(X_source, feature_spec):
    total = 0
    for group, cfg in feature_spec.items():
        width = X_source[group].shape[1]
        method = cfg.get("method", "raw")
        if method == "pca":
            total += min(int(cfg["n_components"]), X_source.shape[0], width)
        else:
            total += width
    return total


comparison_rows = []
forecast_frame = pd.DataFrame({"actual": y}, index=dates)

for source_name in MACRO_SOURCES:
    fred_md = macro_panels[source_name]
    X_source = build_feature_frame(fred_md)
    macro_missing_cells = int(fred_md.isna().sum().sum())

    for feature_method in FEATURE_METHODS:
        feature_spec = FEATURE_SPECS[feature_method]
        forecast_name = f"{source_name}__{feature_method}"
        effective_cols = effective_feature_count(X_source, feature_spec)

        start_time = time.perf_counter()
        model = build_model(ENGINE, feature_spec)
        y_forecast = wu.expanding_window(
            model,
            X_source,
            y,
            dates,
            OOS_START,
            gap=WINDOW_GAP,
            refit_freq=1,
            progress=True,
            tqdm_desc=f"{source_name} | {feature_method}",
        )
        runtime_seconds = time.perf_counter() - start_time

        forecast_frame[forecast_name] = y_forecast
        comparison_rows.append({
            "macro_source": source_name,
            "feature_method": feature_method,
            "macro_cols": fred_md.shape[1],
            "macro_missing_cells": macro_missing_cells,
            "feature_cols_raw": X_source.shape[1],
            "feature_cols_effective": effective_cols,
            "lgbm_grid_size": len(LGBM_REG_GRID),
            "R2_OOS_hist_mean": wu.oos_r2(y, y_forecast, benchmark="hist_mean", gap=WINDOW_GAP),
            "R2_OOS_zero": wu.oos_r2(y, y_forecast, benchmark="zero", gap=WINDOW_GAP),
            "runtime_seconds": runtime_seconds,
        })

comparison = pd.DataFrame(comparison_rows).set_index(["macro_source", "feature_method"]).sort_index()
comparison.round(4)


In [ ]:
plot_df = forecast_frame.loc[forecast_frame.index >= OOS_START].copy()
plot_df = plot_df.rename(
    columns={
        name: name.replace("__", " | ")
        for name in plot_df.columns
        if name != "actual"
    }
)

ax = plot_df.plot(figsize=(11, 4), linewidth=1.1)
ax.set_title(f"{ENGINE.upper()} comparison: {TARGET_MODE} {TARGET_MATURITY}m excess returns")
ax.set_xlabel("Date")
ax.set_ylabel("Excess return")
ax.axhline(0.0, color="grey", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()
